In [ ]:
import requests, pandas as pd, time
from io import StringIO

BASE_URL = "https://data.cdc.gov/resource/vbim-akqf.csv"

select_columns = ",".join([
    "cdc_report_dt",
    "age_group",
    "sex",
    "race_ethnicity_combined",
    "hosp_yn",
    "icu_yn",
    "death_yn",
    "medcond_yn",
    "current_status"
])

TARGET_ROWS = 1_000_000
limit = 200_000
offset = 0
chunks = []

while offset < TARGET_ROWS:
    print(f"Downloading rows {offset} to {offset + limit}...")
    params = {
        "$select": select_columns, 
        "$limit": limit, 
        "$offset": offset}
    
    r = requests.get(BASE_URL, params=params)

    if r.status_code != 200:
        print("Error:", r.status_code, r.text[:300])
        break

    chunk = pd.read_csv(StringIO(r.text))
    
    if chunk.empty:
        break

    chunks.append(chunk)
    offset += limit
    time.sleep(1)

df = pd.concat(chunks, ignore_index=True)
df.to_parquet("C:/Users/gurba/Downloads/cdc-covid-project/data/processed/cdc_sample_1M.parquet", index=False)
print("Saved sample rows:", len(df))

Saved sample rows: 1000000


In [8]:
import pandas as pd

df = pd.read_parquet("C:/Users/gurba/Downloads/cdc-covid-project/data/processed/cdc_sample_1M.parquet")
df.shape

(1000000, 9)

In [9]:
df["age_group"].value_counts(dropna=False).head(20)
df["hosp_yn"].value_counts(dropna=False)
df["icu_yn"].value_counts(dropna=False)
df["death_yn"].value_counts(dropna=False)
df["medcond_yn"].value_counts(dropna=False)

medcond_yn
Missing    805183
Unknown     92369
No          58667
Yes         43781
Name: count, dtype: int64

In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 9 columns):
 #   Column                   Non-Null Count    Dtype
---  ------                   --------------    -----
 0   cdc_report_dt            949569 non-null   str  
 1   age_group                1000000 non-null  str  
 2   sex                      1000000 non-null  str  
 3   race_ethnicity_combined  1000000 non-null  str  
 4   hosp_yn                  1000000 non-null  str  
 5   icu_yn                   1000000 non-null  str  
 6   death_yn                 1000000 non-null  str  
 7   medcond_yn               1000000 non-null  str  
 8   current_status           1000000 non-null  str  
dtypes: str(9)
memory usage: 167.4 MB


In [11]:
for col in df.columns:
    print("\n", col)
    print(df[col].unique()[:10])


 cdc_report_dt
<ArrowStringArray>
['2021-12-30T00:00:00.000', '2024-02-01T00:00:00.000',
 '2022-08-25T00:00:00.000', '2022-01-12T00:00:00.000',
 '2024-02-02T00:00:00.000', '2022-06-28T00:00:00.000',
 '2021-12-31T00:00:00.000', '2022-01-03T00:00:00.000',
 '2024-03-12T00:00:00.000', '2022-08-05T00:00:00.000']
Length: 10, dtype: str

 age_group
<ArrowStringArray>
[  '0 - 9 Years', '60 - 69 Years', '10 - 19 Years', '20 - 29 Years',
 '40 - 49 Years', '50 - 59 Years', '70 - 79 Years']
Length: 7, dtype: str

 sex
<ArrowStringArray>
['Male', 'Unknown', 'Female', 'Missing', 'Other']
Length: 5, dtype: str

 race_ethnicity_combined
<ArrowStringArray>
[                                             'Unknown',
          'American Indian/Alaska Native, Non-Hispanic',
                                              'Missing',
                                  'Black, Non-Hispanic',
                         'Multiple/Other, Non-Hispanic',
                                      'Hispanic/Latino',
         

In [14]:
import pandas as pd

df_clean = df.copy()

# 1) Convert cdc_report_dt to datetime
df_clean["cdc_report_dt"] = pd.to_datetime(df_clean["cdc_report_dt"], errors="coerce")

# 2) Create month column (only works after datetime conversion)
df_clean["case_month"] = df_clean["cdc_report_dt"].dt.to_period("M").astype(str)

# 3) Keep only confirmed + probable cases (matches your dataset values exactly)
df_clean = df_clean[df_clean["current_status"].isin(["Laboratory-confirmed case", "Probable Case"])]

# 4) Create a known-hospitalization subset for the "known-only" denominator
df_hosp_known = df_clean[df_clean["hosp_yn"].isin(["Yes", "No"])]

df_clean[["cdc_report_dt", "case_month", "current_status"]].head()

,cdc_report_dt,case_month,current_status
0,2021-12-30,2021-12,Laboratory-confirmed case
1,2024-02-01,2024-02,Probable Case
2,2022-08-25,2022-08,Laboratory-confirmed case
3,2022-01-12,2022-01,Laboratory-confirmed case
4,2024-02-02,2024-02,Laboratory-confirmed case
